In [30]:
from datasets import load_dataset

# Downloads the dataset and loads it into memory


ds_art = load_dataset("qiaojin/PubMedQA", "pqa_artificial")

ds_unlabel = load_dataset("qiaojin/PubMedQA", "pqa_unlabeled")

ds_label = load_dataset("qiaojin/PubMedQA", "pqa_labeled")


In [31]:
ds_art

DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 211269
    })
})

In [32]:
print(f"type of ds_art: {type(ds_art)}")
print(f"type of ds_unlabel: {type(ds_unlabel)}")
print(f"type of ds_label: {type(ds_label)}")


art_rows = ds_art.num_rows
unlabel_rows = ds_unlabel.num_rows
label_rows = ds_label.num_rows
print("dsadsa",art_rows,unlabel_rows, label_rows)

print(type(ds_art['train']), type(ds_unlabel['train']), type(ds_label['train']))





type of ds_art: <class 'datasets.dataset_dict.DatasetDict'>
type of ds_unlabel: <class 'datasets.dataset_dict.DatasetDict'>
type of ds_label: <class 'datasets.dataset_dict.DatasetDict'>
dsadsa {'train': 211269} {'train': 61249} {'train': 1000}
<class 'datasets.arrow_dataset.Dataset'> <class 'datasets.arrow_dataset.Dataset'> <class 'datasets.arrow_dataset.Dataset'>


In [33]:
len(ds_art['train'])  // 1000

211

In [34]:
print(type(ds_art['train'][0]))

keys = ds_art['train'][0].keys()


print(f"Keys: {keys}")

for i in keys:
    
    print(f"Type: {type(ds_art['train'][0][i])}")
    print(f"{i}: {ds_art['train'][0][i]}\n")



<class 'dict'>
Keys: dict_keys(['pubid', 'question', 'context', 'long_answer', 'final_decision'])
Type: <class 'int'>
pubid: 25429730

Type: <class 'str'>
question: Are group 2 innate lymphoid cells ( ILC2s ) increased in chronic rhinosinusitis with nasal polyps or eosinophilia?

Type: <class 'dict'>
context: {'contexts': ['Chronic rhinosinusitis (CRS) is a heterogeneous disease with an uncertain pathogenesis. Group 2 innate lymphoid cells (ILC2s) represent a recently discovered cell population which has been implicated in driving Th2 inflammation in CRS; however, their relationship with clinical disease characteristics has yet to be investigated.', 'The aim of this study was to identify ILC2s in sinus mucosa in patients with CRS and controls and compare ILC2s across characteristics of disease.', 'A cross-sectional study of patients with CRS undergoing endoscopic sinus surgery was conducted. Sinus mucosal biopsies were obtained during surgery and control tissue from patients undergoing

In [35]:
ds_art['train'][0]


{'pubid': 25429730,
 'question': 'Are group 2 innate lymphoid cells ( ILC2s ) increased in chronic rhinosinusitis with nasal polyps or eosinophilia?',
 'context': {'contexts': ['Chronic rhinosinusitis (CRS) is a heterogeneous disease with an uncertain pathogenesis. Group 2 innate lymphoid cells (ILC2s) represent a recently discovered cell population which has been implicated in driving Th2 inflammation in CRS; however, their relationship with clinical disease characteristics has yet to be investigated.',
   'The aim of this study was to identify ILC2s in sinus mucosa in patients with CRS and controls and compare ILC2s across characteristics of disease.',
   'A cross-sectional study of patients with CRS undergoing endoscopic sinus surgery was conducted. Sinus mucosal biopsies were obtained during surgery and control tissue from patients undergoing pituitary tumour resection through transphenoidal approach. ILC2s were identified as CD45(+) Lin(-) CD127(+) CD4(-) CD8(-) CRTH2(CD294)(+) CD

In [36]:
def make_prompt(example: dict) -> str:
    question = example["question"]
    answer = example["final_decision"]
    context = example["context"]["contexts"]

    return f"""
        Use the medical context below to answer the question.

        Context:
        {context}

        Question:
        {example["question"]}

        Respond with exactly one of these labels:
        yes
        no
        maybe

        Answer:
    """

make_prompt(ds_art['train'][0])



"\n        Use the medical context below to answer the question.\n\n        Context:\n        ['Chronic rhinosinusitis (CRS) is a heterogeneous disease with an uncertain pathogenesis. Group 2 innate lymphoid cells (ILC2s) represent a recently discovered cell population which has been implicated in driving Th2 inflammation in CRS; however, their relationship with clinical disease characteristics has yet to be investigated.', 'The aim of this study was to identify ILC2s in sinus mucosa in patients with CRS and controls and compare ILC2s across characteristics of disease.', 'A cross-sectional study of patients with CRS undergoing endoscopic sinus surgery was conducted. Sinus mucosal biopsies were obtained during surgery and control tissue from patients undergoing pituitary tumour resection through transphenoidal approach. ILC2s were identified as CD45(+) Lin(-) CD127(+) CD4(-) CD8(-) CRTH2(CD294)(+) CD161(+) cells in single cell suspensions through flow cytometry. ILC2 frequencies, measur

In [37]:
ds_art

DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 211269
    })
})

In [38]:
print(f"type: {type(ds_art['train'])}")

print(f"{len(ds_art['train'])}")


type: <class 'datasets.arrow_dataset.Dataset'>
211269


In [56]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from peft import LoraConfig
from main2 import load_data
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported


ds_art, ds_unlabel, ds_label = load_data()

ds_art['train'] = ds_art["train"].select(range(100))

def format_prompt_function(example):
    question = example["question"]
    answer = example["final_decision"]
    
    context = ""

    for i in example["context"]["contexts"]:
        context += i
    
    example["text"] = f"""You are an assistant helping doctors with their questions. You are given the question and the important context you need to answer that question.
    Question: {question}
    Context: {context}
    answer: {answer} """
    
    # print(f"example: {example}")
    return example


    
ds_art["train"] = ds_art["train"].map(format_prompt_function, batched = False)


print(f"type: {type(ds_art)}")
print(f"test: {ds_art['train'][1]['text']}")
max_seq_length = 2048
dtype = None
load_in_4bit = True



# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = "unsloth/Llama-3.2-3B-Instruct", # unsloth/Llama-3.2-3B-Instruct #Qwen/Qwen3-4B
#     max_seq_length=max_seq_length,
#     dtype = dtype,
#     load_in_4bit =load_in_4bit
    
# )

# print(f"Model is loaded and Tokenizer is loaded")

# model = FastLanguageModel.get_peft_model( model,
#     r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
#     target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
#                       "gate_proj", "up_proj", "down_proj",],
#     lora_alpha = 16, # a higher alpha value assigns more weight to the LoRA activations
#     lora_dropout = 0, # Supports any, but = 0 is optimized
#     bias = "none",    # Supports any, but = "none" is optimized
#     use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
#     random_state = 3407,
#     use_rslora = False,
#     loftq_config = None,
# )


# EOS_Token = tokenizer.eos_token

# trainer = SFTTrainer(model = model, 
#                      train_dataset = ds_art,
#                      dataset_text_field = "text",
#                      max_seq_length = max_seq_length,
#                      dataset_num_proc = 24,
#                     args = TrainingArguments(
#                         per_device_train_batch_size = 20, # The batch size per GPU/TPU core
#                         gradient_accumulation_steps = 4, # Number of steps to perform befor each gradient accumulation
#                         warmup_steps = 5, # Few updates with low learning rate before actual training
#                         max_steps = 1, # Specifies the total number of training steps (batches) to run.
#                         learning_rate = 2e-4,
#                         fp16 = not is_bfloat16_supported(),
#                         bf16 = is_bfloat16_supported(),
#                         logging_steps = 1,
#                         optim = "adamw_8bit", # Optimizer
#                         weight_decay = 0.01,
#                         lr_scheduler_type = "linear",
#                         seed = 3407,
#                         output_dir = "outputs",
#                         report_to = "none", # Use this for WandB etc for observability
#                     ),
#                 )

# trainer_stats = trainer.train()

Loaded Datasets: DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 211269
    })
}), DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer'],
        num_rows: 61249
    })
}), DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 1000
    })
})


Map: 100%|██████████| 100/100 [00:00<00:00, 8109.64 examples/s]

type: <class 'datasets.dataset_dict.DatasetDict'>
test: You are an assistant helping doctors with their questions. You are given the question and the important context you need to answer that question.
    Question: Does vagus nerve contribute to the development of steatohepatitis and obesity in phosphatidylethanolamine N-methyltransferase deficient mice?
    Context: Phosphatidylethanolamine N-methyltransferase (PEMT), a liver enriched enzyme, is responsible for approximately one third of hepatic phosphatidylcholine biosynthesis. When fed a high-fat diet (HFD), Pemt(-/-) mice are protected from HF-induced obesity; however, they develop steatohepatitis. The vagus nerve relays signals between liver and brain that regulate peripheral adiposity and pancreas function. Here we explore a possible role of the hepatic branch of the vagus nerve in the development of diet induced obesity and steatohepatitis in Pemt(-/-) mice.8-week old Pemt(-/-) and Pemt(+/+) mice were subjected to hepatic vagot

In [24]:
ds_art['train']

ValueError: Column 'train' doesn't exist.